# Semantic Search using FAISS + Embeddings

**FAISS: Facebook AI Similarity Search**
```
FAISS finds similarity using:
- L2 Distance (Euclidean distance)
- Cosine similarity (via normalization)
```

## Objective (Explain to students)

1) Convert text → embeddings

2) Store in vector database using FAISS

3) Perform semantic search

In [ ]:
# Following too long time
# !pip install faiss-cpu sentence-transformers

# SO I used 
# !python -m pip install faiss-cpu sentence-transformers

In [21]:
# Import Librarairs: Took a 4-5 minutes
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load Embedding Model

In [50]:
# Pretrained embedding model

model = SentenceTransformer('all-MiniLM-L6-v2') # works. Total parameters ≈ 22.7 Million
# model = SentenceTransformer('paraphrase-MiniLM-L3-v2') # works: Total parameters ≈ 17.39 Million
# model = SentenceTransformer('BAAI/bge-small-en') # works. Total parameters ≈ 33.36 Million

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- This model converts text → vectors
- Each sentence becomes a numeric representation

# Lets check the model size

In [51]:
params = sum(p.numel() for p in model.parameters())

# Each parameter ~4 bytes (float32)
size_mb = params * 4 / (1024 * 1024)

print(f"Total parameters ≈ {params/1e6:.2f} Million")
print(f"Model size ≈ {size_mb:.2f} MB")

Total parameters ≈ 22.71 Million
Model size ≈ 86.64 MB


# Where is the model stored locally

In [43]:
import os
import pathlib
from sentence_transformers import SentenceTransformer

model_path = model._first_module().auto_model.config._name_or_path
print("model_path:", model_path)

# If downloaded locally, check cache folder
cache_dir = pathlib.Path.home() / ".cache" / "huggingface"
print("cache_dir:", cache_dir)

model_path: BAAI/bge-small-en
cache_dir: C:\Users\hi\.cache\huggingface


# Create Sample Dataset

In [44]:
documents = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Python is used for data science",
    "Cars and vehicles are transportation",
    "Artificial intelligence is transforming industries",
    "Football is a popular sport",
    "Data engineering involves pipelines",
    "Big data tools include Hadoop and Spark"
]

# Convert Text → Embeddings

In [45]:
embeddings = model.encode(documents)
print("Shape of embeddings:", embeddings.shape)

Shape of embeddings: (8, 384)


- Each sentence → vector of numbers
- Shape = (num_docs, vector_size)

# Create FAISS Index

In [46]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance: Euclidean distance (similarity measure)
index.add(embeddings)

print("Number of vectors in DB:", index.ntotal)

Number of vectors in DB: 8


- FAISS stores vectors
- Enables fast similarity search

# Perform Semantic Search

In [47]:
query = "AI is changing the world"
query_embedding = model.encode([query])

k = 3  # top results
distances, indices = index.search(query_embedding, k)

print("Top results:")
for i in indices[0]:
    print(documents[i])

Top results:
Artificial intelligence is transforming industries
Machine learning is a subset of AI
Deep learning uses neural networks


# Add more realistic data

In [48]:
documents.extend([
    "Chatbots are powered by large language models",
    "Healthcare is improved by AI diagnostics. It is bringing positive change in the world",
    "Self-driving cars use machine learning"
])

embeddings = model.encode(documents)
print("Shape of embeddings:", embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance (similarity measure)
index.add(embeddings)

print("Number of vectors in DB:", index.ntotal)

Shape of embeddings: (11, 384)
Number of vectors in DB: 11


In [49]:
query = "AI is changing the world"
query_embedding = model.encode([query])

k = 3  # top results
distances, indices = index.search(query_embedding, k)

print("Top results:")
for i in indices[0]:
    print(documents[i])

Top results:
Artificial intelligence is transforming industries
Healthcare is improved by AI diagnostics. It is bringing positive change in the world
Machine learning is a subset of AI
